IMPORTING LIBRARIES

In [83]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.documents import Document
import numpy as np
import os
import json
from dotenv import load_dotenv
load_dotenv()

True

API DETAILS

In [35]:
api_key = os.getenv('OPENAI_API_KEY')

EXTRACTING DATA - REPAIR MANUAL

In [12]:
document_loader = PyPDFLoader('/Users/niharikareddy/Desktop/Assignments/Udemy/ML_Projects/GIT_HUB/Intelligent_Vehicle_Diagnostics_Copilot/data/manuals/repair_manual.pdf').load()

CHUNKING DATA - REPAIR MANUAL

In [67]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(document_loader)

In [68]:
for chunk in chunks:
    chunk.metadata["source"] = "repair_manual"

EXTRACTING DATA - OWNER MANUAL

In [69]:
camry_document_loader = PyPDFLoader('/Users/niharikareddy/Desktop/Assignments/Udemy/ML_Projects/GIT_HUB/Intelligent_Vehicle_Diagnostics_Copilot/data/manuals/2018-camry.pdf').load()

CHUNKING DATA - OWNER MANUAL

In [70]:
camry_chunks = text_splitter.split_documents(camry_document_loader)

In [71]:
for chunk in camry_chunks:
    chunk.metadata["source"] = "owner_manual"

COMBINING CHUNKS

In [85]:
all_chunks_retriever = camry_chunks + chunks

In [97]:
all_chunks_retriever

[Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows)', 'creator': 'FrameMaker 2015.1', 'creationdate': '2017-03-16T08:52:16+00:00', 'author': '129647', 'moddate': '2017-10-16T11:31:25-05:00', 'title': 'CAMRY_OM_North_America_OM06122U.book', 'source': 'owner_manual', 'total_pages': 612, 'page': 0, 'page_label': '1'}, page_content='CAMRY_U (01999-06122)\nPictorial index Search by illustration\n1 For safety \nand security Make sure to read through them\n2 Instrument \ncluster\nHow to read the gauges and meters, the variety of \nwarning lights and indicators, etc.\n3\nOperation of \neach \ncomponent\nOpening and closing the doors and windows, \nadjustment before driving, etc.\n4 Driving Operations and advice which are necessary for \ndriving\n5 Interior features Usage of the interior features, etc.\n6 Maintenance \nand care\nCaring for your vehicle and maintenance \nprocedures\n7 When trouble \narises What to do in case of malfunction or emergency\n8 Vehicle \nspecifications\n

In [ ]:
with open("./data/formatted_obd.json") as f:
    obd_data = json.load(f)

obd_docs = [
    Document(
        page_content=f"OBD Code {item['code']}: {item['description']}",
        metadata={"source": "obd_codes", "system": item["system"]}
    )
    for item in obd_data
]

all_chunks = all_chunks_retriever + obd_docs

EMBEDDINGS

In [87]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectors = FAISS.from_documents(
    documents=all_chunks,
    embedding=embeddings,
)

RETRIEVER

In [125]:
bm25_retriever = BM25Retriever.from_documents(all_chunks)
bm25_retriever.k = 2

faiss_retriever = vectors.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 30, "lambda_mult": 0.7}
)

retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.7, 0.3]
)

In [126]:
def format_docs(docs):
    return "\n\n".join([
        f"[Source: {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in docs
    ])


LLM CREATION

In [122]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key)

PROMPT TEMPLATE

In [131]:
hyde_prompt = ChatPromptTemplate.from_template("""
Write a passage from a vehicle repair manual or owner's manual that would directly answer this question.
Include technical details such as symptoms, causes, diagnostic steps, or repair procedures if relevant.

Question: {question}
Passage:
""")

hyde_chain = hyde_prompt | llm | StrOutputParser()

prompt = ChatPromptTemplate.from_template("""
You are an expert vehicle diagnostics assistant helping mechanics and car owners diagnose and fix vehicle issues.
Answer the question using ONLY the provided context from the vehicle manuals.
Focus only on mechanical and combustion-related causes. 
Ignore unrelated issues such as battery, immobilizer, or starting problems.
- Differentiate between:Standard OBD-II codes and Manufacturer-specific codes
If multiple OBD codes are relevant, provide a range instead of a single example.
Rules:
- Label them clearly.
- Always mention the specific system (engine, transmission, brakes, etc.) involved
- Include OBD codes, torque specs, or part numbers if available in the context
- If diagnostic steps are available, list them in order
- Only say "This information is not available in the provided manuals" if the context has absolutely no relevant information — do not add it if you have already given an answer
- Do not make assumptions or use external knowledge outside the provided context
- If safety is a concern, clearly flag it

Context:
{context}

Question: {question}

Answer STRICTLY in this format:

System:
- ...
                                                                                   
Possible Causes:
- ...

Recommended Checks:
- ...

Related Error Codes:
- ...

Severity:
- Low / Medium / High

Sources:
- ..

Confidence Score:
- 0.xx  
                                                                                                                                                      
Answer:                                        
""")

document_chain = (
    RunnableParallel(
        context=RunnableLambda(lambda x: hyde_chain.invoke({"question": x["question"]})) | retriever | RunnableLambda(format_docs),
        question=RunnableLambda(lambda x: x["question"])
    )
    | prompt | llm | StrOutputParser()
)


In [132]:
test_queries = [
    "What are the causes and fixes for engine misfire?",
    "Why does a car overheat and how to fix it?",
    "Why is my car not starting?"
]
for i in test_queries:
    ans = document_chain.invoke({"question": i})
    print(f"Q: {i}\nA: {ans}\n")

Q: What are the causes and fixes for engine misfire?
A: System:
- Engine

Possible Causes:
- Contamination in the fuel system from large or small particles that have bypassed fuel filters.
- Air filter blockage leading to insufficient air entering the combustion chamber, causing misfire issues.

Recommended Checks:
1. Inspect the fuel filter and replace if it is clogged or hasn't been changed at the recommended mileage.
2. Inspect the air filter for dirt and debris and replace if necessary.
3. Check for any other signs of fuel or air delivery issues.

Related Error Codes:
- P0301: Cylinder 1 Misfire Detected (and potentially other cylinder misfire codes such as P0302, P0303...)

Severity:
- Medium

Sources:
- [Source: obd_codes]
- [Source: repair_manual]  

Confidence Score:
- 0.90  

Q: Why does a car overheat and how to fix it?
A: System:
- Cooling System

Possible Causes:
- Insufficient coolant in the system
- Thermostat malfunction
- Coolant leaks in the cooling system
- Blocked ra